# 322. Coin Change

## Topic Alignment
- This unbounded knapsack problem models resource optimization scenarios like change-making in financial systems, packet assembly in networking, or selecting hyperparameters from continuous ranges in ML pipelines.

## Metadata Summary
- Source: https://leetcode.com/problems/coin-change/
- Tags: Dynamic Programming, Array, Complete Knapsack, BFS
- Difficulty: Medium
- Priority: High

## Problem Statement
You are given an integer array `coins` representing coins of different denominations and an integer `amount` representing a total amount of money.

Return the **fewest number of coins** that you need to make up that amount. If that amount of money cannot be made up by any combination of the coins, return `-1`.

You may assume that you have an **infinite number** of each kind of coin.

**Constraints**:
- 1 <= coins.length <= 12
- 1 <= coins[i] <= 2^31 - 1
- 0 <= amount <= 10^4

## Progressive Hints
- Hint 1: This is the classic **Complete Knapsack** problem - each coin can be used unlimited times.
- Hint 2: Use dp[i] to represent the minimum coins needed to make amount i.
- Hint 3: For each amount, try using each coin and take the minimum.
- Hint 4: Traverse amount from left to right (allowing coin reuse in current iteration).
- Hint 5: Initialize dp[0] = 0 (0 coins for amount 0), others to infinity.

## Solution Overview
This is the canonical **Complete Knapsack (Unbounded Knapsack)** problem:
- **Items**: coins of different denominations
- **Capacity**: target amount
- **Key property**: Each coin can be used **unlimited times**
- **Goal**: Minimize number of coins to reach target amount

**Approaches**:
1. **BFS**: Treat as shortest path problem, each coin is an edge
2. **Top-down DP + memo**: Recursive with caching
3. **Bottom-up DP**: Iterative, most efficient

**State**: `dp[i]` = minimum coins needed to make amount i

**Recurrence**: 
```python
dp[i] = min(dp[i], dp[i - coin] + 1) for all valid coins
```

## Detailed Explanation

### Understanding Complete Knapsack

**0/1 Knapsack** (each item used at most once):
- Traverse capacity **right to left**
- Ensures we use previous iteration's values
- Example: LC 416 Partition Equal Subset Sum

**Complete Knapsack** (each item used unlimited times):
- Traverse capacity **left to right**
- Allows using current iteration's already-updated values
- Example: This problem (LC 322 Coin Change)

---

### Why Traverse Left to Right?

**Goal**: Allow using the same coin multiple times.

**Complete Knapsack (left to right)**:
```python
for coin in coins:
    for amount in range(coin, target + 1):  # Left to right
        dp[amount] = min(dp[amount], dp[amount - coin] + 1)
```
- When we update `dp[amount]`, we use `dp[amount - coin]` which might have been **updated** in this iteration
- If `dp[amount - coin]` was updated, it means we already used `coin` to reach `amount - coin`
- Now we use `coin` again to reach `amount`
- This allows **unlimited reuse** of the same coin!

**Example**: coins = [1, 2], amount = 5
- When processing coin=2:
  - dp[2] = dp[0] + 1 = 1 (use one coin of 2)
  - dp[4] = dp[2] + 1 = 2 (use coin=2 again! dp[2] was just updated)
  - dp[5] = dp[3] + 1 = ...

**0/1 Knapsack would be** (right to left):
```python
for coin in coins:
    for amount in range(target, coin - 1, -1):  # Right to left
        dp[amount] = min(dp[amount], dp[amount - coin] + 1)
```
- Uses **previous iteration's** values only
- Each coin used at most once

---

### Complete vs 0/1 Knapsack: Loop Order Comparison

**Complete Knapsack** (this problem):
```python
# Outer loop: items (coins)
for coin in coins:
    # Inner loop: capacity (amount), LEFT TO RIGHT
    for amount in range(coin, target + 1):
        dp[amount] = min(dp[amount], dp[amount - coin] + 1)
```
- **Effect**: Can use same coin multiple times
- **Use case**: Unlimited supply of items

**0/1 Knapsack** (LC 416):
```python
# Outer loop: items
for item in items:
    # Inner loop: capacity, RIGHT TO LEFT
    for capacity in range(target, item_weight - 1, -1):
        dp[capacity] = max(dp[capacity], dp[capacity - item_weight] + item_value)
```
- **Effect**: Each item used at most once
- **Use case**: Limited supply (one of each item)

---

### Algorithm Steps

**Step 1**: Initialize DP array
- `dp[0] = 0` (need 0 coins to make amount 0)
- `dp[i] = inf` for i > 0 (initially impossible)

**Step 2**: For each coin denomination:
- For each amount from coin to target (left to right!):
  - If we can make (amount - coin), we can make amount
  - Update: `dp[amount] = min(dp[amount], dp[amount - coin] + 1)`

**Step 3**: Return result
- If `dp[target] == inf`, return -1 (impossible)
- Otherwise, return `dp[target]`

---

### Example Walkthrough

**Input**: coins = [1, 2, 5], amount = 11

**Initial**: `dp = [0, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf, inf]`

**After coin = 1**:
- dp[1] = dp[0] + 1 = 1
- dp[2] = dp[1] + 1 = 2
- ...
- dp[11] = dp[10] + 1 = 11
- `dp = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]`

**After coin = 2**:
- dp[2] = min(2, dp[0] + 1) = 1
- dp[3] = min(3, dp[1] + 1) = 2
- dp[4] = min(4, dp[2] + 1) = 2 (using coin=2 twice!)
- ...
- `dp = [0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6]`

**After coin = 5**:
- dp[5] = min(3, dp[0] + 1) = 1
- dp[6] = min(3, dp[1] + 1) = 2
- dp[10] = min(5, dp[5] + 1) = 2 (5+5)
- dp[11] = min(6, dp[6] + 1) = 3 (5+5+1 or 5+2+2+2 or other combinations)
- `dp = [0, 1, 1, 2, 2, 1, 2, 2, 3, 3, 2, 3]`

**Result**: dp[11] = 3 (one way: 5+5+1)

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Backtracking | O(amount^n) | O(amount) | Exponential, too slow |
| BFS | O(amount × n) | O(amount) | Treat as shortest path |
| Top-down DP + memo | O(amount × n) | O(amount) | Recursive with cache |
| Bottom-up DP | O(amount × n) | O(amount) | Most efficient, recommended |

In [ ]:
class Solution:
    def coinChange(self, coins: list[int], amount: int) -> int:
        """
        Complete Knapsack DP solution.
        
        Time: O(amount × n) where n = len(coins)
        Space: O(amount)
        """
        # dp[i] = minimum coins needed to make amount i
        dp = [float('inf')] * (amount + 1)
        dp[0] = 0  # Base case: 0 coins for amount 0
        
        # Complete knapsack: outer loop items, inner loop capacity LEFT TO RIGHT
        for coin in coins:
            for amt in range(coin, amount + 1):
                # If we can make (amt - coin), we can make amt
                if dp[amt - coin] != float('inf'):
                    dp[amt] = min(dp[amt], dp[amt - coin] + 1)
        
        # Return result
        return dp[amount] if dp[amount] != float('inf') else -1

In [ ]:
# Test cases
tests = [
    ([1, 2, 5], 11, 3),        # 5+5+1
    ([2], 3, -1),              # Impossible
    ([1], 0, 0),               # Amount 0
    ([1], 1, 1),               # Single coin
    ([1], 2, 2),               # Use same coin twice
    ([1, 2, 5], 100, 20),      # 5×20
    ([186, 419, 83, 408], 6249, 20),  # Complex case
    ([1, 3, 4], 6, 2),         # 3+3
]

solver = Solution()
for coins, amount, expected in tests:
    result = solver.coinChange(coins, amount)
    assert result == expected, f"Failed for coins={coins}, amount={amount}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(amount × n) where n = len(coins)
  - For each coin (n iterations), we iterate through amounts from coin to target
  - Total iterations: approximately n × amount
- **Space**: O(amount)
  - Single DP array of size amount + 1
  - No additional space needed

## Edge Cases & Pitfalls
- **amount = 0**: Return 0 immediately (need 0 coins)
- **Impossible case**: If no combination works, return -1 (check dp[amount] == inf)
- **Single coin**: If coins = [x] and amount % x == 0, return amount / x
- **Greedy doesn't work**: coins = [1, 3, 4], amount = 6. Greedy gives 4+1+1=3 coins, but optimal is 3+3=2 coins
- **Initialization**: dp[0] = 0 is crucial; all others should be inf, not -1 or 0
- **Overflow**: Use float('inf') for infinity, not sys.maxint
- **Loop direction**: MUST traverse left to right for complete knapsack
- **Coin value larger than amount**: Automatically skipped by loop range

## Follow-up Variants
- **Count combinations**: How many ways to make the amount? (LC 518)
- **Count permutations**: Treat different orderings as different ways (LC 377)
- **Limited coins**: Each coin has limited quantity (becomes 0/1 knapsack variant)
- **Maximum amount**: Given k coins, what's the maximum amount you can make?
- **Coin denominations design**: Design optimal coin denominations for a currency
- **Weighted coins**: Each coin has different weight, minimize weight instead of count

## Takeaways
- **Complete Knapsack Pattern**: Traverse capacity left to right to allow item reuse
- **Minimization DP**: Use `min(dp[i], dp[i-coin] + 1)` to find minimum coins
- **Infinity Initialization**: Use float('inf') for impossible states, not -1 or 0
- **Greedy Fails**: This problem cannot be solved greedily; always use largest coin doesn't guarantee minimum count
- **Loop Order Matters**: Left-to-right (complete) vs right-to-left (0/1) determines whether items can be reused
- **Template Reusability**: This complete knapsack template applies to many similar problems (LC 518, 377, 279)

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 518 | Coin Change II | Complete knapsack counting combinations |
| LC 377 | Combination Sum IV | Complete knapsack counting permutations |
| LC 279 | Perfect Squares | Complete knapsack minimization |
| LC 983 | Minimum Cost For Tickets | DP with multiple choices |
| LC 39 | Combination Sum | Backtracking + DP variant |